# 20 — Coding Agent Devbox: Claude Code in an AgentCore Runtime microVM

This notebook drives **`orgiq-devbox`** — a tenth runtime that hosts the
**Claude Code CLI** instead of a Strands agent. It demonstrates the AgentCore
Runtime preview features end-to-end:

- **Managed session storage** (preview): a persistent `/mnt/workspace` that
  survives stop/resume of the microVM
- **`InvokeAgentRuntimeCommand`**: one-shot deterministic shell in the
  container, streamed stdout/stderr — no LLM in the loop
- **Interactive shell** (`agentcore exec --it`) from the JupyterLab Terminal

```
┌────────────────────────────┐
│ JupyterLab (this notebook) │
│                            │
│  cell ── boto3 ─────────┐  │        ┌──────────────────────────────┐
│   (task mode)           │  │        │  AgentCore Runtime microVM   │
│  cell ── boto3 ─────────┼──┼──────▶ │  ┌────────────────────────┐  │
│   (command mode)        │  │        │  │ main.py wrapper (:8080)│  │
│                         │  │        │  │   └─▶ claude -p ...    │  │
│  Terminal tab ──────────┘  │        │  └────────────────────────┘  │
│   agentcore exec --it      │        │  /mnt/workspace (persistent) │
└────────────────────────────┘        │  IAM role ──▶ Bedrock Claude │
                                      └──────────────────────────────┘
```

**Auth chain — no API keys anywhere.** This notebook uses your host SSO
credentials → AgentCore data plane → the microVM runs under the devbox
execution role → Claude Code (`CLAUDE_CODE_USE_BEDROCK=1`) calls Bedrock
inference with those role credentials.

## Prerequisites

1. Build + push the devbox image (one-time, and after any wrapper change):
   ```bash
   ./agents/devbox/build.sh
   ```
2. Deploy the runtime:
   ```bash
   cd infra && ./scripts/deploy.sh OrgIQ-Devbox --context deployDevbox=true
   ```

## ⚠️ Session-storage lifecycle (read before relying on the workspace)

| Event | Effect on `/mnt/workspace` |
|---|---|
| Stop + resume same session ID | **Survives** — that's the demo below |
| Runtime version update (any image push + deploy) | **Wiped** |
| 14 days without an invocation on the session | **Wiped** |
| `DeleteAgentRuntime` / `destroy.sh` | **Wiped** |

Hard limits: 1 GB per session, 8 h max session lifetime, session IDs ≥ 33
chars. Treat the workspace as a cache — durable work leaves via git or the
command API.

In [ ]:
import json
import time
import uuid
import boto3
import botocore
from pathlib import Path
from packaging.version import Version

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
ACCOUNT_ID = config["ACCOUNT_ID"]

# InvokeAgentRuntimeCommand ships in botocore >= 1.42.92
assert Version(botocore.__version__) >= Version("1.42.92"), (
    f"botocore {botocore.__version__} is too old for the devbox command API — "
    "rebuild the jupyter image (requirements.txt pins boto3>=1.42.92)"
)

agentcore = boto3.client("bedrock-agentcore", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)

# Resolve the devbox runtime ARN from the OrgIQ-Devbox stack export.
exports = {e["Name"]: e["Value"] for page in cfn.get_paginator("list_exports").paginate() for e in page["Exports"]}
DEVBOX_ARN = exports.get(f"{PREFIX}-devbox-runtime-arn")
assert DEVBOX_ARN, (
    f"Export {PREFIX}-devbox-runtime-arn not found — deploy with "
    "./scripts/deploy.sh OrgIQ-Devbox --context deployDevbox=true"
)

# Session IDs must be >= 33 chars. Reuse across notebook runs via config.json
# so the persistent workspace demo composes across sessions.
SESSION_ID = config.get("DEVBOX_SESSION_ID") or f"devbox-{uuid.uuid4().hex}{uuid.uuid4().hex[:8]}"
config["DEVBOX_SESSION_ID"] = SESSION_ID
Path("config.json").write_text(json.dumps(config, indent=2))

print(f"✓ Devbox runtime: {DEVBOX_ARN}")
print(f"✓ Session ID ({len(SESSION_ID)} chars): {SESSION_ID}")

## 1. Command mode — first contact (no LLM, no cost)

`InvokeAgentRuntimeCommand` executes a shell command **directly in the
microVM** — it never touches the wrapper or Claude Code. Output streams back
as `contentDelta` chunks with an `exitCode` on `contentStop`. Perfect for
proving the plumbing works before spending any Bedrock tokens.

The first call on a fresh session ID provisions the microVM (cold start
~10–30 s); subsequent calls in the same session are fast.

In [ ]:
import shlex

def run_command(command: str, timeout: int = 300) -> int:
    """Run a shell command in the devbox microVM; stream output; return exit code.

    The command API exec's argv directly — no shell (verified live: `cd`
    fails with "executable file not found"). It does apply shell-style word
    splitting with quote support, so wrap in `sh -c <quoted>` to get shell
    semantics (&&, cd, globs).
    """
    # HOME matches task mode's (pinned to the workspace); GIT_CONFIG_* injects
    # safe.directory per-invocation — the mount is root-owned while commands
    # run as UID 1000, so git otherwise fails with "dubious ownership".
    full = ("export HOME=/mnt/workspace GIT_CONFIG_COUNT=1 "
            "GIT_CONFIG_KEY_0=safe.directory GIT_CONFIG_VALUE_0=/mnt/workspace; "
            + command)
    resp = agentcore.invoke_agent_runtime_command(
        agentRuntimeArn=DEVBOX_ARN,
        runtimeSessionId=SESSION_ID,
        body={"command": f"sh -c {shlex.quote(full)}", "timeout": timeout},
    )
    exit_code = None
    for event in resp["stream"]:
        chunk = event.get("chunk", {})
        if "contentDelta" in chunk:
            delta = chunk["contentDelta"]
            if delta.get("stdout"):
                print(delta["stdout"], end="")
            if delta.get("stderr"):
                print(delta["stderr"], end="")
        elif "contentStop" in chunk:
            stop = chunk["contentStop"]
            exit_code = stop.get("exitCode")
            print(f"\n[exit {exit_code} — {stop.get('status')}]")
    return exit_code

run_command("uname -a && echo && node --version && claude --version && echo && ls -la /mnt/workspace")

## 2. Task mode — Claude Code writes a project

Task mode goes through the wrapper: `InvokeAgentRuntime` → `claude -p
<prompt>` headless in `/mnt/workspace`, with `stream-json` events streamed
back as SSE. The wrapper also points `HOME` at the workspace so Claude Code's
own conversation state persists alongside the files.

In [ ]:
def run_task(prompt: str, resume: bool = False, timeout: int = 840) -> None:
    """Invoke Claude Code in the devbox; render the stream-json event stream."""
    resp = agentcore.invoke_agent_runtime(
        agentRuntimeArn=DEVBOX_ARN,
        runtimeSessionId=SESSION_ID,
        qualifier="DEFAULT",
        payload=json.dumps({"prompt": prompt, "resume": resume, "timeout": timeout}).encode(),
    )
    body = resp["response"]
    for raw in body.iter_lines():
        line = raw.decode("utf-8", errors="replace").strip()
        if line.startswith("data:"):
            line = line[5:].strip()
        if not line or line[0] not in "{\"":
            continue
        try:
            ev = json.loads(line)
            if isinstance(ev, str):  # tolerate double-encoded frames
                ev = json.loads(ev)
        except json.JSONDecodeError:
            continue
        if not isinstance(ev, dict):
            continue
        et = ev.get("type")
        if et == "devbox_start":
            mode = "persistent" if ev.get("workspace_persistent") else "EPHEMERAL (no mount!)"
            print(f"── workspace {ev.get('workspace')} [{mode}] · model {ev.get('model')}")
        elif et == "system" and ev.get("subtype") == "init":
            print(f"── claude session {ev.get('session_id', '')[:8]}… started")
        elif et == "assistant":
            for block in ev.get("message", {}).get("content", []):
                if block.get("type") == "text" and block.get("text", "").strip():
                    print(block["text"])
                elif block.get("type") == "tool_use":
                    arg = json.dumps(block.get("input", {}))[:100]
                    print(f"  ⚙ {block.get('name')} {arg}")
        elif et == "result":
            print(f"── result: {ev.get('subtype')} · {ev.get('num_turns')} turns · "
                  f"{ev.get('duration_ms', 0) / 1000:.0f}s · ${ev.get('total_cost_usd', 0):.4f}")
        elif et == "devbox_end":
            print(f"── exit {ev.get('exit_code')}" + (" (TIMED OUT)" if ev.get("timed_out") else ""))
            if ev.get("exit_code") not in (0, None) and ev.get("stderr"):
                print(ev["stderr"])

run_task(
    "Scaffold a small Python project in the current directory: a calculator "
    "module with add and subtract functions, pytest tests for both, and a "
    "README. Initialize a git repo and commit everything with a sensible message."
)

### Verify deterministically — the LLM's work checked without the LLM

In [ ]:
run_command("cd /mnt/workspace && python -m pytest -q 2>&1; echo; git log --oneline; echo; ls -la")

## 3. The money demo — *"it's safe to close your laptop now"*

Stop the session: the microVM is torn down entirely. Resume with the **same
session ID**: a fresh microVM comes up, session storage is re-mounted, and
both the working tree **and** Claude Code's conversation history (`~/.claude`
lives on the mount) are exactly where they were left.

> Always wait for `stop_runtime_session` to complete before resuming — the
> flush to durable storage happens during graceful shutdown.

In [ ]:
print("Stopping session (tearing down the microVM)…")
agentcore.stop_runtime_session(
    agentRuntimeArn=DEVBOX_ARN,
    runtimeSessionId=SESSION_ID,
)
time.sleep(10)  # let the stop + flush complete
print("Stopped. Resuming the same session — files AND conversation should be intact:\n")

# resume=True → `claude --continue`: Claude Code picks up its own prior
# conversation, so "the script you created" resolves with no re-explanation.
run_task(
    "Add a multiply function to the calculator module you created, with a test. "
    "Commit the change.",
    resume=True,
)

In [ ]:
# Old commits + the new one, on a microVM that did not exist a minute ago:
run_command("cd /mnt/workspace && git log --oneline && python -m pytest -q 2>&1")

## 4. Interactive terminal — a real PTY into the microVM

The bidirectional shell (`InvokeAgentRuntimeCommandShell`) is not in botocore
— Python can't speak bidirectional streams — so it **cannot run in a notebook
cell**. It runs from the **JupyterLab Terminal** instead (File ▸ New ▸
Terminal); the `agentcore` CLI is baked into this image.

```bash
# Interactive shell into the live devbox session (max 1 h per connection,
# up to 10 concurrent shells per runtime):
agentcore exec --it --runtime orgiq_devbox --session-id $DEVBOX_SESSION_ID

# Reconnect to a shell you were disconnected from:
agentcore exec --it --session-id $DEVBOX_SESSION_ID --shell-id <shell-id>
```

Get the session ID into your terminal environment first:

```bash
export DEVBOX_SESSION_ID=$(python -c "import json; print(json.load(open('/workspace/aws/config.json'))['DEVBOX_SESSION_ID'])")
```

Inside the shell you can `cd /mnt/workspace` and run `claude` **fully
interactively** — same workspace, same Bedrock auth, colors and Ctrl-C and
all.

## 5. Cleanup

The devbox session costs nothing while stopped — compute only runs during
invocations, and session storage is free in preview. Still:

- **Stop the session** below when done experimenting.
- The runtime itself is **CDK-owned** (`OrgIQ-Devbox`) — it is *not*
  registered with the notebook `ResourceTracker`, and `19-cleanup` won't touch
  it. Tear it down with `infra/scripts/destroy.sh`, or redeploy without
  `--context deployDevbox=true` to remove just the runtime.
- To start over with a blank workspace: delete `DEVBOX_SESSION_ID` from
  `config.json` and re-run the setup cell (new session = fresh storage).

In [ ]:
agentcore.stop_runtime_session(
    agentRuntimeArn=DEVBOX_ARN,
    runtimeSessionId=SESSION_ID,
)
print(f"✓ Session {SESSION_ID[:20]}… stopped. Workspace persists for 14 idle days.")